# Text Summarization

Text summarization is the process of creating a short, accurate, and fluent summary of a longer text document.

In this notebook, we will implement an **Extractive Text Summarization** approach. This method involves selecting the most important sentences from the original text and concatenating them to form a summary.

### Steps:
1. **Text Cleaning:** Remove stopwords, punctuation, etc.
2. **Word Frequency Calculation:** Calculate how often each word appears.
3. **Sentence Scoring:** Score sentences based on the frequency of the words they contain.
4. **Summary Generation:** Pick the top-ranked sentences.

In [3]:
import nltk
import re
import heapq  # For finding the top N sentences

# Download necessary NLTK datasets
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\chaud\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\chaud\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Input Text
We will use a sample text about Artificial Intelligence.

In [4]:
text = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural intelligence displayed by animals including humans.
Leading AI textbooks define the field as the study of "intelligent agents": any system that perceives its environment and takes actions that maximize its chance of achieving its goals.
Some popular accounts use the term "artificial intelligence" to describe machines that mimic "cognitive" functions that humans associate with the human mind, such as "learning" and "problem solving".
AI applications include advanced web search engines (e.g., Google), recommendation systems (used by YouTube, Amazon and Netflix), understanding human speech (such as Siri and Alexa), self-driving cars (e.g., Tesla), automated decision-making and competing at the highest level in strategic game systems (such as chess and Go).
As machines become increasingly capable, tasks considered to require "intelligence" are often removed from the definition of AI, a phenomenon known as the AI effect.
For instance, optical character recognition is frequently excluded from things considered to be AI, having become a routine technology.
Artificial intelligence was founded as an academic discipline in 1956, and in the years since has experienced several waves of optimism, followed by disappointment and the loss of funding (known as an "AI winter"), followed by new approaches, success and renewed funding.
AI research has tried and discarded many different approaches since its founding, including simulating the brain, modeling human problem solving, formal logic, large databases of knowledge and imitating animal behavior.
In the first decades of the 21st century, highly mathematical statistical machine learning has dominated the field, and this technique has proved highly successful, helping to solve many challenging problems throughout industry and academia.
"""

### 1. Preprocessing and Word Frequency
We calculate the frequency of each word, ignoring stopwords (common words like 'the', 'is', 'and' that carry less meaning).

In [5]:
# Tokenize the text into sentences
sentences = nltk.sent_tokenize(text)

# Get English stopwords
stopwords = nltk.corpus.stopwords.words('english')

# Calculate Word Frequencies
word_frequencies = {}
for word in nltk.word_tokenize(text):
    if word.lower() not in stopwords:
        if word.lower() not in word_frequencies.keys():
            word_frequencies[word.lower()] = 1
        else:
            word_frequencies[word.lower()] += 1

print(f"Total unique words (excluding stopwords): {len(word_frequencies)}")

Total unique words (excluding stopwords): 154


### 2. Weighted Frequencies
We normalize the frequencies by dividing by the maximum frequency. This gives us a weight between 0 and 1 for each word.

In [6]:
maximum_frequency = max(word_frequencies.values())

for word in word_frequencies.keys():
    word_frequencies[word] = (word_frequencies[word] / maximum_frequency)

# Check a few word weights
print("Word weights sample:", list(word_frequencies.items())[:5])

Word weights sample: [('artificial', 0.125), ('intelligence', 0.25), ('(', 0.2916666666666667), ('ai', 0.3333333333333333), (')', 0.2916666666666667)]


### 3. Sentence Scoring
We calculate a score for each sentence by summing up the weights of the words it contains.

In [7]:
sentence_scores = {}

for sent in sentences:
    for word in nltk.word_tokenize(sent.lower()):
        if word in word_frequencies.keys():
            # We can also add a constraint for sentence length here if needed
            # For example, ignoring very long sentences: if len(sent.split(' ')) < 30:
            if sent not in sentence_scores.keys():
                sentence_scores[sent] = word_frequencies[word]
            else:
                sentence_scores[sent] += word_frequencies[word]

# Print scores for the first few sentences
for sent in list(sentence_scores.keys())[:3]:
    print(f"Score: {sentence_scores[sent]:.2f} | Sentence: {sent[:50]}...")

Score: 3.67 | Sentence: 
Artificial intelligence (AI) is intelligence demo...
Score: 2.04 | Sentence: Leading AI textbooks define the field as the study...
Score: 5.08 | Sentence: Some popular accounts use the term "artificial int...


### 4. Generate Summary
We select the top N sentences with the highest scores to form the summary.

In [8]:
# Select the top 3 sentences
summary_sentences = heapq.nlargest(3, sentence_scores, key=sentence_scores.get)

summary = ' '.join(summary_sentences)

print("Original Text Length:", len(text))
print("Summary Length:", len(summary))
print("\n--- SUMMARY ---\n")
print(summary)

Original Text Length: 1898
Summary Length: 818

--- SUMMARY ---

AI applications include advanced web search engines (e.g., Google), recommendation systems (used by YouTube, Amazon and Netflix), understanding human speech (such as Siri and Alexa), self-driving cars (e.g., Tesla), automated decision-making and competing at the highest level in strategic game systems (such as chess and Go). Artificial intelligence was founded as an academic discipline in 1956, and in the years since has experienced several waves of optimism, followed by disappointment and the loss of funding (known as an "AI winter"), followed by new approaches, success and renewed funding. AI research has tried and discarded many different approaches since its founding, including simulating the brain, modeling human problem solving, formal logic, large databases of knowledge and imitating animal behavior.
